# Python Data Manipulation Practice (Google Product DS)

These Python questions strictly follow the `SQL-python-prob-draft.md` rule format.

### Problem 1: Consecutive Login Streak
**Description**: Given a dataframe of user logins, find the maximum consecutive days each user has logged in.

**Sample Input**:
| user_id | login_date |
|---------|------------|
| 1       | 2023-01-01 |
| 1       | 2023-01-02 |
| 1       | 2023-01-04 |
| 2       | 2023-01-01 |
| 2       | 2023-01-02 |
| 2       | 2023-01-03 |

**Sample Output**:
| user_id | max_streak |
|---------|------------|
| 1       | 2          |
| 2       | 3          |

In [4]:
import pandas as pd
import numpy as np

# Dataframe Creation
data = {
    'user_id': [1, 1, 1, 2, 2, 2, 3, 3, 3, 4, 4, 4, 5, 5],
    'login_date': [
        '2023-01-01', '2023-01-02', '2023-01-04', 
        '2023-01-01', '2023-01-02', '2023-01-03',
        '2023-01-05', '2023-01-06', '2023-01-08',
        '2023-01-01', '2023-01-03', '2023-01-05',
        '2023-01-10', '2023-01-11'
    ]
}
df = pd.DataFrame(data)
df['login_date'] = pd.to_datetime(df['login_date'])
df.head(5)

,user_id,login_date
0,1,2023-01-01
1,1,2023-01-02
2,1,2023-01-04
3,2,2023-01-01
4,2,2023-01-02


In [29]:
df_cleaned = df.drop_duplicates().sort_values(by=['user_id','login_date'],ascending=[True,True])
df_cleaned['rank'] = df.groupby('user_id').cumcount()

df_cleaned['group'] = df['login_date'] - pd.to_timedelta(df_cleaned['rank'],unit='D')

df_cleaned['streak'] = df_cleaned.groupby('user_id')['group'].cumcount()

# df_streaks = df_cleaned.groupby('user_id')['streak'].max().reset_index()
df_cleaned
# df_streaks , type(df_streaks)

,user_id,login_date,rank,group,streak
0,1,2023-01-01,0,2023-01-01,0
1,1,2023-01-02,1,2023-01-01,1
2,1,2023-01-04,2,2023-01-02,2
3,2,2023-01-01,0,2023-01-01,0
4,2,2023-01-02,1,2023-01-01,1
5,2,2023-01-03,2,2023-01-01,2
6,3,2023-01-05,0,2023-01-05,0
7,3,2023-01-06,1,2023-01-05,1
8,3,2023-01-08,2,2023-01-06,2
9,4,2023-01-01,0,2023-01-01,0


**Tips for problem-solving**:
1. Sort the dataframe by `user_id` and `login_date`.
2. Drop exact duplicate rows (a user logging in twice on the same day counts as 1 day in the streak).
3. Use the concept of `date - row_number` (or difference between consecutive dates) to group consecutive dates together.
4. Group by the `user_id` and the newly created consecutive group, then find the size of each group.
5. Finally, find the max streak per user.

In [10]:
df


,user_id,login_date
0,1,2023-01-01
1,1,2023-01-02
2,1,2023-01-04
3,2,2023-01-01
4,2,2023-01-02
5,2,2023-01-03


In [5]:
# Optimized Solution
def max_login_streak(df):
    # Sort and drop duplicates for same-day logins
    df_clean = df.drop_duplicates().sort_values(by=['user_id', 'login_date']).copy()
    
    # Create a grouping key: if dates are consecutive, subtracting a daily incrementing timedelta yields a constant date
    df_clean['rank'] = df_clean.groupby('user_id').cumcount()
    df_clean['grp_date'] = df_clean['login_date'] - pd.to_timedelta(df_clean['rank'], unit='D')
    
    # Aggregate to find lengths of consecutive groups
    # streak_lengths = df_clean.groupby(['user_id', 'grp_date']).size().reset_index(name='streak')
    
    # Find maximum streak per user
    # max_streaks = streak_lengths.groupby('user_id')['streak'].max().reset_index(name='max_streak')
    
    return df_clean

max_login_streak(df)

,user_id,login_date,rank,grp_date
0,1,2023-01-01,0,2023-01-01
1,1,2023-01-02,1,2023-01-01
2,1,2023-01-04,2,2023-01-02
3,2,2023-01-01,0,2023-01-01
4,2,2023-01-02,1,2023-01-01
5,2,2023-01-03,2,2023-01-01


### Problem 2: Analyzing Session CTR
**Description**: You have a dataframe of user sessions where a session can have multiple 'impression' and 'click' events. Calculate the overall app CTR, and return a robust metric handling potential division by zero.

**Sample Input**:
| session_id | event_type |
|------------|------------|
| s1         | impression |
| s1         | click      |
| s2         | impression |
| s3         | click      |

**Sample Output**:
| ctr  |
|------|
| 1.0  |

In [ ]:
# Dataframe Creation
events = pd.DataFrame({
    'session_id': ['s1', 's1', 's2', 's3'],
    'event_type': ['impression', 'click', 'impression', 'click']
})
events

**Tips for problem-solving**:
1. Count total clicks and impressions.
2. Be sure to handle scenarios where impressions are 0 (e.g. data logging errors) to avoid `ZeroDivisionError` or returning `inf`.
3. Use `numpy.where` or basic conditional logic.

In [ ]:
# Optimized Solution
def safe_ctr(df):
    clicks = (df['event_type'] == 'click').sum()
    impressions = (df['event_type'] == 'impression').sum()
    
    # Safe division
    ctr = clicks / impressions if impressions > 0 else np.nan
    return pd.DataFrame({'ctr': [ctr]})

safe_ctr(events)